# DMP-05 validated composite

This validated composite keeps every original notebook cell and adds only a local statsmodels OLS fallback plus additive validation checks.

In [ ]:
# Local statsmodels.api.OLS fallback for validated execution only.
try:
    import statsmodels.api as _statsmodels_check
except Exception:
    import sys
    import types
    import numpy as np
    import pandas as pd

    class _OLSResult:
        def __init__(self, beta):
            self.params = pd.Series([beta])

    class _OLS:
        def __init__(self, y, x):
            self.y = y
            self.x = x
        def fit(self):
            df = pd.concat([pd.Series(self.y), pd.Series(self.x)], axis=1).dropna()
            yy = df.iloc[:, 0].to_numpy()
            xx = df.iloc[:, 1].to_numpy()
            beta = float(np.dot(xx, yy) / np.dot(xx, xx))
            return _OLSResult(beta)

    statsmodels = types.ModuleType("statsmodels")
    api = types.ModuleType("statsmodels.api")
    api.OLS = _OLS
    statsmodels.api = api
    sys.modules["statsmodels"] = statsmodels
    sys.modules["statsmodels.api"] = api
    print("Installed local statsmodels.api.OLS fallback for DMP-05 validation.")


# QuantiP — Building a Quant Strategy in Python (DMP-05)
### EPAT · Week 25-1 · instructor **Ishan Shah**

This notebook reproduces the **whole QuantiP lecture on real, shipped Nifty-50 data** — no synthetic
series and no live downloads. Everything runs offline on the files that ship with the class:

* `nifty_stocks_prices.csv` — daily close for **48 Nifty-50 stocks + `^NSEI`** (the index), 2013-01-01 → 2025-06-19
* `nifty50_list.csv` — the constituent list with each firm's **Return-on-Equity (ROE)**
* the shipped **modules** `alpha.py`, `data.py`, `performance_analytics.py` (imported, not re-typed)

**Part 1 — the professional workflow:** Data → Screening → Alpha → Backtest → Performance, plus
modular programming and the single biggest lever, diversification.
**Part 2 — a real academic anomaly:** systematic vs idiosyncratic risk, **beta**, and
**Betting Against Beta** (Frazzini & Pedersen) with a Quality (ROE) overlay.


## Setup — load the real data and the shipped modules

The lecture's whole point about *modular programming* is that data access and signal logic live in their
own files. We import them exactly as the shipped notebooks do.

In [ ]:
import warnings, os, sys
warnings.simplefilter("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# the shipped modules sit next to this notebook
sys.path.insert(0, os.path.abspath("."))
from alpha import ma_crossover, compute_adx           # signal logic (EWM crossover + ADX)
from performance_analytics import compute_ret         # ret & strategy_ret = ret * signal.shift(1)

pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25, "font.size": 10})
AMBER, ORANGE, BROWN, STONE = "#b45309", "#d97706", "#78350f", "#6b7280"

# --- price matrix: 48 stocks + the index ---
prices = pd.read_csv("nifty_stocks_prices.csv", index_col=0)
prices.index = pd.to_datetime(prices.index)
# --- constituent list with ROE ---
nifty_list = pd.read_csv("nifty50_list.csv", index_col=0)

print("prices :", prices.shape, "|", prices.index[0].date(), "->", prices.index[-1].date())
print("columns:", ", ".join([c for c in prices.columns[:6]]), "... + '^NSEI'")
print("list   :", nifty_list.shape, "| columns:", list(nifty_list.columns))
nifty_list.head(3)

## Part 1 — The professional strategy workflow

A professional never *randomly starts coding*. There is a fixed sequence, and most of the value lives in
the steps beginners skip:

| # | Step | What it means |
|---|------|---------------|
| 1 | **Data** | price · fundamentals · sentiment · legal/economic |
| 2 | **Screening** | drop penny / illiquid / over-volatile / recent-M&A names |
| 3 | **Alpha** | signals that beat the benchmark; combine several |
| 4 | **Backtest** | shift the signal, include transaction cost, no look-ahead |
| 5 | **Performance** | judge on Sharpe, drawdown and vol — *not* return alone |

Sobering benchmark: **~90 % of active managers trail their index.** If your strategy can't beat a cheap
ETF, it has no reason to exist.

### 1 · Data & modular programming

Four families of data feed alpha — **price** (OHLCV), **fundamentals** (revenue, profit, earnings *dates*),
**sentiment** (company con-calls → news → social, e.g. Reddit/GameStop) and **legal/economic** (war, the Fed,
even counting ships through the Strait of Hormuz to forecast crude).

The programming lesson: don't paste `yf.download(...)` into a hundred notebooks. Wrap it in a **module**
(`get_stock_data()` in `data.py`). The real payoff isn't fewer lines — it's **maintenance**: if the data
vendor dies you change *one* function, not a hundred files. (`auto_adjust=True` fixes splits/bonuses; it does
**not** "average" prices.)

In [ ]:
# The shipped modules we just imported ARE the modular-programming lesson:
import inspect, alpha, performance_analytics
print("alpha.py exposes      :", [f for f in dir(alpha) if not f.startswith('_') and callable(getattr(alpha,f))])
print("perf_analytics exposes:", [f for f in dir(performance_analytics) if not f.startswith('_') and callable(getattr(performance_analytics,f))])
print()
print("ma_crossover signature:", inspect.signature(alpha.ma_crossover))
print("compute_ret  signature:", inspect.signature(performance_analytics.compute_ret))
print()
print("Universe loaded from one CSV — 48 tradable names + the index, all offline:")
print(", ".join([c for c in prices.columns if c != '^NSEI']))

### 2 · Screening — the liquidity step everyone forgets

Throw away what doesn't fit: **penny** stocks (circuit-locked, beginner magnets), **over-volatile** names
(if your mandate forbids them), and recent **mergers** (the merged entity ≠ its history). But the screen
**90 % of people forget is liquidity**: you can absorb only about **1 % of a stock's daily dollar volume**
without moving the price against yourself. Work the lecture's example.

In [ ]:
# Lecture's liquidity worked example
adv_shares  = 1.18e6      # average daily shares traded
price       = 12.6        # share price
daily_dollar = adv_shares * price
absorbable   = 0.01 * daily_dollar      # 1% rule
want_to_deploy = 10_000_000

print(f"avg daily dollar volume = 1.18M shares x ${price} = ${daily_dollar:,.0f}")
print(f"1% absorbable per day               = ${absorbable:,.0f}")
print(f"capital we want to deploy           = ${want_to_deploy:,.0f}")
print("-"*52)
print("VERDICT:", "AVOID  (would take too long / slippage brutal)" if absorbable < want_to_deploy else "OK")

### 3 · Alpha — a testable hypothesis (MA crossover)

Ideas come from four wells: **own experience**, **others' experience**, your **network**, and **academic
research** (SSRN anomalies). Whatever the idea, turn it into a *testable hypothesis*. "If it rises two days
it keeps rising" → **buy when EMA(2) > EMA(7)**. We use the shipped `alpha.ma_crossover`, applied to a real
Nifty name (`RELIANCE`) straight from the shipped price matrix.

In [ ]:
# Single-stock demo on RELIANCE (real shipped close data)
STOCK = "RELIANCE"
d = pd.DataFrame({"Close": prices[STOCK].dropna()})

short_lookback, long_lookback = 2, 7
d = ma_crossover(d, short_lookback, long_lookback)   # adds ma_short, ma_long, ma_signal (EWM)

print(f"{STOCK}: EMA({short_lookback}) vs EMA({long_lookback}) crossover")
print(d[["Close","ma_short","ma_long","ma_signal"]].dropna().head())
print("\nbars long (signal=1):", int(d.ma_signal.sum()), "of", int(d.ma_signal.notna().sum()),
      f"({d.ma_signal.mean()*100:.0f}% of bars)")

### 4 & 5 · Combining alphas, backtesting with cost, judging honestly

One alpha is rarely enough. The shipped framework confirms the trend with **ADX > 20** and combines with an
**AND** (multiply the two 0/1 signals). ADX needs High/Low (see `alpha.compute_adx`), but the shipped price
*matrix* is close-only — so here we use a **close-based trend filter** (price above its 50-day average) as
the offline stand-in and keep the exact combine-and-backtest discipline: **position = signal shifted one bar**
(no look-ahead) and **transaction cost on every trade**.

In [ ]:
# Alpha 1 = MA crossover (ma_signal). Alpha 2 = trend filter (Close > 50-day SMA).
d["trend_signal"] = np.where(d["Close"] > d["Close"].rolling(50).mean(), 1, 0)

# --- (a) raw crossover alone ---
d["signal"] = d["ma_signal"]
d = compute_ret(d)                                    # ret + strategy_ret = ret * signal.shift(1)
cum_raw = (1 + d["strategy_ret"].fillna(0)).cumprod()
bh      = (1 + d["ret"].fillna(0)).cumprod()
sharpe_raw = d.strategy_ret.mean()/d.strategy_ret.std()*np.sqrt(252)
flips_raw  = int(d["signal"].fillna(0).diff().abs().eq(1).sum())

# --- (b) combined: crossover AND trend ---
d["signal"] = (d["ma_signal"] * d["trend_signal"]).fillna(0)
d = compute_ret(d)
cum_cmb = (1 + d["strategy_ret"].fillna(0)).cumprod()
sharpe_cmb = d.strategy_ret.mean()/d.strategy_ret.std()*np.sqrt(252)
flips_cmb  = int(d["signal"].diff().abs().eq(1).sum())

print(f"raw MA(2/7) crossover : cum {(cum_raw.iloc[-1]-1)*100:6.1f}% | Sharpe {sharpe_raw:4.2f} | signal-changes {flips_raw}")
print(f"buy & hold {STOCK:<10}: cum {(bh.iloc[-1]-1)*100:6.1f}%")
print(f"crossover AND trend   : cum {(cum_cmb.iloc[-1]-1)*100:6.1f}% | Sharpe {sharpe_cmb:4.2f} | signal-changes {flips_cmb}")
print("\nHonest read: a fast unfiltered crossover OVER-TRADES ({} flips) and trails buy-&-hold.".format(flips_raw))
print("That is exactly why the lecture adds a trend filter (ADX) and, next, DIVERSIFIES.")

#### Backtest with transaction cost — the trade log

No backtest is honest without cost. The shipped framework charges **brokerage 0.03 % + tax 0.02 % = 0.05 %
per side** and rebuilds a trade-by-trade log. We reproduce it for the combined strategy.

In [ ]:
def get_trades(data, close_col, signal_col):
    trades, cur, entry = [], 0, None
    for i in data.index:
        nw = data.loc[i, signal_col]
        if nw != cur:
            if entry is not None:
                trades.append((cur, entry, data.loc[entry, close_col], i, data.loc[i, close_col]))
                entry = None
            if nw != 0:
                entry = i
            cur = nw
    tr = pd.DataFrame(trades, columns=["Position","Entry Time","Entry Price","Exit Time","Exit Price"])
    tr["PnL"] = (tr["Exit Price"] - tr["Entry Price"]) * tr["Position"]
    return tr

trades = get_trades(d, "Close", "signal")
gross = trades.PnL.sum()

brokerage, tax = 0.03/100, 0.02/100
total_cost = brokerage + tax
trades["PnL"] -= total_cost * (trades["Entry Price"] + trades["Exit Price"])   # qty = 1
net = trades.PnL.sum()

print(f"transaction cost per side : {total_cost*100:.3f}%")
print(f"number of trades          : {len(trades)}")
print(f"gross PnL (points)        : {gross:8.1f}")
print(f"net PnL after costs       : {net:8.1f}   (cost drag {(gross-net)/gross*100:.0f}%)")
trades.head()

### The biggest lever in Part 1 — diversification

Beginners reach for stop-losses and parameter sweeps. The simplest, strongest move is to run the **same**
strategy across a **basket of loosely-correlated names**. Winners cover for losers, portfolio volatility
falls, and the Sharpe ratio jumps. We apply the shipped `ma_crossover` to **every** stock in the matrix and
build an equally-weighted portfolio.

In [ ]:
strat = {}
single_sharpes = []
for tk in [c for c in prices.columns if c != "^NSEI"]:
    s = prices[tk].dropna()
    if len(s) < 300:
        continue
    x = pd.DataFrame({"Close": s})
    x = ma_crossover(x, 2, 7)
    x["signal"] = x["ma_signal"]
    x = compute_ret(x)
    strat["ret_" + tk] = x["strategy_ret"]
    single_sharpes.append(x.strategy_ret.mean()/x.strategy_ret.std()*np.sqrt(252))

portfolio = pd.DataFrame(strat)
portfolio["portfolio"] = portfolio.mean(axis=1)                 # equal-weight
port_sharpe = portfolio.portfolio.mean()/portfolio.portfolio.std()*np.sqrt(252)
port_cum    = (1 + portfolio.portfolio.fillna(0)).cumprod()

print(f"stocks in basket            : {len(single_sharpes)}")
print(f"AVG single-stock Sharpe     : {np.mean(single_sharpes):.2f}")
print(f"equal-weight PORTFOLIO Sharpe: {port_sharpe:.2f}   (~{port_sharpe/np.mean(single_sharpes):.1f}x higher)")
print(f"portfolio cumulative return : {(port_cum.iloc[-1]-1)*100:.1f}%")
print("\nSame strategy, many names -> volatility falls, Sharpe roughly doubles. That is the lever.")

## Part 2 — Betting Against Beta

### Two kinds of risk

* **Idiosyncratic (unsystematic) risk** is firm-specific — a strike at one Maruti plant hurts Maruti, not
  Hyundai. You can **diversify it away**, so the market does **not** pay you for it.
* **Systematic risk** is market-wide — COVID, war, 2008, a Fed decision — it hits everything at once. You
  can't diversify it, only hedge it, so it **is** compensated. We measure it with **beta**.

### Beta — the slope of stock returns on market returns

$$R^{stock}_i = \alpha + \beta\, R^{market}_i$$

Beta is the OLS slope. We use the shipped BAB recipe: daily returns, then regress each stock on `^NSEI`.

In [ ]:
dpc = prices.pct_change()[2:]          # daily returns (shipped BAB uses [2:])

def calc_beta(y, x):
    df = pd.concat([y, x], axis=1).dropna()
    return sm.OLS(df.iloc[:, 0], df.iloc[:, 1]).fit().params.iloc[0]

# two contrasting names, estimated on the training window (<= 2017)
b_tcs   = calc_beta(dpc.loc[:'2017', "TCS"],       dpc.loc[:'2017', "^NSEI"])
b_steel = calc_beta(dpc.loc[:'2017', "TATASTEEL"], dpc.loc[:'2017', "^NSEI"])
print(f"beta  TCS       = {b_tcs:.2f}   (defensive IT: ~half as volatile as the market)")
print(f"beta  TATASTEEL = {b_steel:.2f}   (cyclical metal: ~1.5x the market)")

print("\nInterpretation ladder:")
for lo, hi, label in [(1.0,99,">1  more volatile than market (e.g. TATASTEEL, banks)"),
                      (0.999,1.001,"=1  moves with the market"),
                      (0,0.999,"0-1 less volatile / defensive (TCS, HINDUNILVR)"),
                      (-99,0,"<0  negatively correlated (Gold, VIX)")]:
    print(f"  beta {label}")

#### Beta for every Nifty-50 name (training window ≤ 2017)

Betas are estimated **only on data up to 2017** — the training window — so that when we trade from 2018 we
are not peeking at the future (look-ahead bias). The cross-section runs roughly **0.46 → 1.76**.

In [ ]:
beta = pd.Series({tk: calc_beta(dpc.loc[:'2017', tk], dpc.loc[:'2017', "^NSEI"])
                  for tk in dpc.columns if tk != "^NSEI"}).sort_values()

print("lowest-beta 6 :"); print(beta.head(6).round(2).to_string())
print("\nhighest-beta 6:"); print(beta.tail(6).round(2).to_string())
print(f"\nrange: {beta.min():.2f} .. {beta.max():.2f}   (^NSEI beta = 1.00 by construction)")

### The anomaly — Betting Against Beta

CAPM says $E[R]=r_f+\beta(R_m-r_f)$; with $r_f\approx0$ that is $E[R]\approx\beta R_m$, so **higher beta
should mean higher return**. Investors chase high beta to juice returns because they are
**leverage-constrained** (retail can't borrow 3×; funds have mandates). That crowding leaves
**high-beta overpriced and low-beta underpriced** — so the contrarian bet is **long low-beta / short
high-beta**.

**Alpha 1:** an equally-weighted portfolio of the low-beta names (**beta < 0.7**), held from **2018 onward**.

In [ ]:
low_beta = beta[beta < 0.7].index.tolist()
print(f"LOW-BETA names (beta < 0.7): {len(low_beta)}")
print(", ".join(low_beta))

def perf(rets):
    cum = (1 + rets).cumprod()
    total = cum.iloc[-1] - 1
    cagr  = (1 + total) ** (252/len(rets)) - 1
    vol   = rets.std() * np.sqrt(252)
    sharpe= rets.mean()/rets.std() * np.sqrt(252)
    mdd   = (cum/cum.cummax() - 1).min()
    return dict(total=total*100, cagr=cagr*100, vol=vol*100, sharpe=sharpe, mdd=mdd*100), cum

lb_ret    = dpc.loc['2018':, low_beta].mean(axis=1)
nifty_ret = dpc.loc['2018':, "^NSEI"]
lb_m,   lb_cum   = perf(lb_ret)
nf_m,   nf_cum   = perf(nifty_ret)
print(f"\nLow-beta   : total {lb_m['total']:.1f}% | CAGR {lb_m['cagr']:.2f}% | vol {lb_m['vol']:.1f}% | Sharpe {lb_m['sharpe']:.2f} | MDD {lb_m['mdd']:.1f}%")
print(f"Nifty 50   : total {nf_m['total']:.1f}% | CAGR {nf_m['cagr']:.2f}% | vol {nf_m['vol']:.1f}% | Sharpe {nf_m['sharpe']:.2f} | MDD {nf_m['mdd']:.1f}%")

**Alpha 2 — add a Quality overlay (ROE).** A colleague's tweak: keep only low-beta names that are also
high-quality, **ROE > 18**. This is the *Quality-minus-Junk* idea in miniature.

In [ ]:
high_roe = nifty_list[nifty_list.ROE > 18].Symbol.values.tolist()
common   = sorted(set(high_roe) & set(low_beta))
print(f"HIGH-ROE names (ROE > 18): {len(high_roe)}")
print(f"LOW-BETA & HIGH-ROE      : {len(common)} -> {common}")

lbroe_ret = dpc.loc['2018':, common].mean(axis=1)
q_m, q_cum = perf(lbroe_ret)
print(f"\nLow-beta & ROE>18: total {q_m['total']:.1f}% | CAGR {q_m['cagr']:.2f}% | vol {q_m['vol']:.1f}% | Sharpe {q_m['sharpe']:.2f} | MDD {q_m['mdd']:.1f}%")

#### Low-beta vs high-beta, head to head — the anomaly in one table

Sort the universe by beta and compare the **lowest-beta five** against the **highest-beta five** over the
same 2018+ window. CAPM predicts high-beta should win. It doesn't: low-beta earns **more** return with a
**higher Sharpe** — the essence of Betting Against Beta.

In [ ]:
p1  = beta.index[:5].tolist()     # lowest beta
p10 = beta.index[-5:].tolist()    # highest beta
p1_m,  _ = perf(dpc.loc['2018':, p1].mean(axis=1))
p10_m, _ = perf(dpc.loc['2018':, p10].mean(axis=1))
print(f"P1 low-beta  ({', '.join(p1)})")
print(f"   beta {beta[p1].min():.2f}-{beta[p1].max():.2f} | total {p1_m['total']:.1f}% | Sharpe {p1_m['sharpe']:.2f}")
print(f"P10 high-beta ({', '.join(p10)})")
print(f"   beta {beta[p10].min():.2f}-{beta[p10].max():.2f} | total {p10_m['total']:.1f}% | Sharpe {p10_m['sharpe']:.2f}")
print(f"\nBAB spread (P1 - P10) = {p1_m['total']-p10_m['total']:+.1f} points, and low-beta did it with LESS risk.")

summary = pd.DataFrame({
    "Low-beta (<0.7)":   lb_m,
    "Low-beta & ROE>18": q_m,
    "Nifty 50":          nf_m,
}).T[["total","cagr","vol","sharpe","mdd"]].round(2)
summary.columns = ["Total %","CAGR %","Vol %","Sharpe","MaxDD %"]
summary

### The two biases that quietly ruin this

* **Survivorship bias** — building the universe from *today's* index members backtests only the winners
  (bankrupt/removed names have vanished). *Fix:* use the index's **historical constituents** at each date
  (Nifty rebalances every six months via NSE press releases).
* **Look-ahead bias** — computing beta on the *full* sample uses data you wouldn't have had. *Fix:* estimate
  beta on a **training window (≤ 2017)** and trade **2018+**, rebalancing periodically — exactly what we did.

**Why low-beta wins in crashes (the recovery math):** a 10 % fall needs an 11 % gain to recover; a 50 % fall
needs a full **100 %**. Low-beta names fall less, so they climb back faster — the engine of BAB's edge.

In [ ]:
for fall in [10, 20, 50, 80]:
    print(f"  a {fall:2d}% drawdown requires a {fall/(100-fall)*100:5.1f}% gain just to break even")

## The QuantiP picture — four panels

1. **Diversification** — the same MA strategy on every name; the bold line is the equal-weight portfolio.
2. **Beta** — TCS vs the Nifty, with the OLS slope (β).
3. **Beta cross-section** — every stock sorted low → high beta.
4. **Betting Against Beta** — low-beta and low-beta-&-quality vs the Nifty, 2018+.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13.2, 9.4))

# --- Panel 1: diversification ---
a = ax[0,0]
ind = portfolio.drop(columns="portfolio")
sub = (1 + ind.fillna(0)).cumprod()
for c in sub.columns:
    a.plot(sub.index, sub[c], color=STONE, lw=.5, alpha=.30)
a.plot(port_cum.index, port_cum, color=AMBER, lw=2.6, label=f"Equal-weight portfolio (Sharpe {port_sharpe:.2f})")
a.set_title(f"1 · Diversification lifts Sharpe {np.mean(single_sharpes):.2f} -> {port_sharpe:.2f}", fontweight="bold")
a.set_ylabel("cumulative growth (x)"); a.legend(fontsize=8, loc="upper left"); a.set_yscale("log")

# --- Panel 2: beta scatter TCS vs Nifty ---
a = ax[0,1]
xr = dpc.loc[:'2017', "^NSEI"]*100; yr = dpc.loc[:'2017', "TCS"]*100
a.scatter(xr, yr, s=6, color=ORANGE, alpha=.35)
xs = np.linspace(xr.min(), xr.max(), 50)
a.plot(xs, b_tcs*xs, color=BROWN, lw=2.2, label=f"OLS slope = beta = {b_tcs:.2f}")
a.axhline(0, color="grey", lw=.6); a.axvline(0, color="grey", lw=.6)
a.set_title("2 · Beta = slope of TCS vs Nifty returns", fontweight="bold")
a.set_xlabel("Nifty 50 daily return (%)"); a.set_ylabel("TCS daily return (%)"); a.legend(fontsize=8)

# --- Panel 3: beta cross-section ---
a = ax[1,0]
colors = ["#16a34a" if b < 0.7 else ("#dc2626" if b > 1.2 else STONE) for b in beta.values]
a.bar(range(len(beta)), beta.values, color=colors)
a.axhline(1.0, color=BROWN, lw=1.2, ls="--", label="market beta = 1.0")
a.axhline(0.7, color="#16a34a", lw=1.0, ls=":", label="low-beta cutoff 0.7")
a.set_title(f"3 · Beta of every Nifty name ({beta.min():.2f}-{beta.max():.2f})", fontweight="bold")
a.set_ylabel("beta (vs ^NSEI, <=2017)"); a.set_xlabel("stocks sorted low -> high beta"); a.legend(fontsize=8)

# --- Panel 4: BAB equity curves ---
a = ax[1,1]
a.plot(lb_cum.index,  lb_cum,  color=AMBER,  lw=2.2, label=f"Low-beta <0.7  (Sharpe {lb_m['sharpe']:.2f})")
a.plot(q_cum.index,   q_cum,   color=BROWN,  lw=2.0, label=f"Low-beta & ROE>18 (Sharpe {q_m['sharpe']:.2f})")
a.plot(nf_cum.index,  nf_cum,  color=STONE,  lw=1.8, ls="--", label=f"Nifty 50  (Sharpe {nf_m['sharpe']:.2f})")
a.set_title("4 · Betting Against Beta beats the index, 2018+", fontweight="bold")
a.set_ylabel("cumulative growth (x)"); a.legend(fontsize=8, loc="upper left")

fig.suptitle("QuantiP — Quant Strategy Workflow & Betting Against Beta (real Nifty-50 data)",
             fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0,0,1,0.97])
fig.savefig("chart_1_quantip.png", dpi=115, bbox_inches="tight")
print("saved chart_1_quantip.png")
plt.show()

## Recap

* A quant strategy is a **process**, not a clever indicator: **Data → Screening → Alpha → Backtest →
  Performance**, in that order.
* **Modular programming** is about *maintenance* — wrap data access so a vendor change touches one file.
* **Screen for liquidity**: you can absorb only ~1 % of daily dollar volume. The lecture's name fails
  (~\$149k absorbable vs \$10 M wanted → **avoid**).
* A fast MA crossover **over-trades** (507 signal changes on RELIANCE) and trails buy-&-hold; the fix is a
  trend filter **and** the quietest, strongest lever — **diversification** (avg single-stock Sharpe
  **0.51 → portfolio 1.28**, ~2.5×).
* **Beta** is the OLS slope of a stock on the market. Our universe runs **0.46–1.76** (TCS 0.52,
  TATASTEEL 1.47).
* **Betting Against Beta**: long low-beta (<0.7, 12 names) returned **+237 %** at Sharpe **1.10** vs the
  Nifty's **+134 %** / **0.76**, with a shallower drawdown (−28 % vs −38 %). Adding a **ROE > 18** quality
  overlay (5 names) lifted it to **+269 %**. Sorted head-to-head, low-beta beat high-beta on **both** return
  and Sharpe — the anomaly.
* Guard against **survivorship** and **look-ahead** bias as fiercely as you guard the alpha — avoiding
  self-deception *is* the edge.

# Additive validation checks

In [ ]:
import pandas as pd
from pathlib import Path
base = Path('.')
files = ['dmp05_source_pdf_inventory.csv', 'dmp05_source_zip_inventory.csv', 'dmp05_data_inventory.csv', 'dmp05_dependency_fallback_scope.csv', 'dmp05_liquidity_strategy_metrics.csv', 'dmp05_trade_cost_metrics.csv', 'dmp05_diversification_metrics.csv', 'dmp05_beta_cross_section.csv', 'dmp05_bab_performance_metrics.csv', 'dmp05_drawdown_recovery_table.csv', 'dmp05_validation_controls.csv']
data = {f: pd.read_csv(base / f) for f in files}
{k: v.shape for k, v in data.items()}

## 1. Source, data, and dependency checks

In [ ]:
print(data['dmp05_source_pdf_inventory.csv'][['file','pages','keyword_hits']].to_string(index=False))
print(data['dmp05_source_zip_inventory.csv'].groupby(['zip','role']).size().to_string())
print(data['dmp05_data_inventory.csv'].to_string(index=False))
print(data['dmp05_dependency_fallback_scope.csv'].to_string(index=False))
prices = data['dmp05_data_inventory.csv'].query("file == 'nifty_stocks_prices.csv'").iloc[0]
assert prices['tradable_stock_columns'] == 47

## 2. Strategy, cost, and diversification checks

In [ ]:
print(data['dmp05_liquidity_strategy_metrics.csv'].to_string(index=False))
print(data['dmp05_trade_cost_metrics.csv'].to_string(index=False))
print(data['dmp05_diversification_metrics.csv'].to_string(index=False))
div = data['dmp05_diversification_metrics.csv'].set_index('metric')
assert div.loc['equal_weight_portfolio_sharpe','value'] > div.loc['avg_single_stock_sharpe','value']

## 3. Beta and BAB checks

In [ ]:
print(data['dmp05_beta_cross_section.csv'].head(8).to_string(index=False))
print(data['dmp05_beta_cross_section.csv'].tail(8).to_string(index=False))
print(data['dmp05_bab_performance_metrics.csv'].to_string(index=False))
print(data['dmp05_drawdown_recovery_table.csv'].to_string(index=False))
bab = data['dmp05_bab_performance_metrics.csv'].set_index('portfolio')
assert bab.loc['Low-beta (<0.7)','sharpe'] > bab.loc['Nifty 50','sharpe']